### Baseline + Classical Machine Learning

Evaluate how accurately classical machine-learning models can predict
Remaining Useful Life (RUL).


In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_dataset

train_df, test_df, rul_df = load_dataset("FD001")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("RUL shape:", rul_df.shape)

Train shape: (20631, 26)
Test shape: (13096, 26)
RUL shape: (100, 1)


Load Dataset

The original FD001 training and test datasets were loaded using the
reusable project data loader.

It also uses the same dataset established in the previous notebook.

In [2]:
RUL_CAP = 125

train_df["max_cycle"] = (
    train_df.groupby("unit")["cycle"].transform("max")
)

train_df["RUL_raw"] = (
    train_df["max_cycle"] - train_df["cycle"]
)

train_df["RUL"] = (
    train_df["RUL_raw"].clip(upper=RUL_CAP)
)

#### Recreate RUL Target

The Phase 1 RUL definition is recreated so that Phase 3 uses the same
prediction target.

The target is capped at 125 cycles.

In [3]:
print("RUL minimum:", train_df["RUL"].min())
print("RUL maximum:", train_df["RUL"].max())
print("RUL missing:", train_df["RUL"].isna().sum())

RUL minimum: 0
RUL maximum: 125
RUL missing: 0


#### Verify RUL Target

The target contains values from 0 to 125 with no missing observations.

This confirms that the intended Phase 1 target is available for machine
learning.

In [4]:
train_df = train_df.sort_values(
    ["unit", "cycle"]
).reset_index(drop=True)

print(
    "Chronologically sorted:",
    train_df.groupby("unit")["cycle"]
    .apply(lambda x: x.is_monotonic_increasing)
    .all()
)

Chronologically sorted: True


#### Preserve Temporal Order

Observations are ordered by engine and cycle to preserve the temporal
structure of each engine trajectory.

In [5]:
sensor_cols = [
    f"sensor_{i}"
    for i in range(1, 22)
]

print("Number of sensors:", len(sensor_cols))

Number of sensors: 21


#### Define Sensor Features

The 21 sensor measurements are used as the initial feature representation.

This provides a simple reference before evaluating more complex feature
representations.

In [6]:
constant_sensors = [
    col for col in sensor_cols
    if train_df[col].nunique() <= 1
]

print("Constant sensors:", constant_sensors)
print("Number of constant sensors:", len(constant_sensors))

Constant sensors: ['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Number of constant sensors: 6


#### Identify Constant Sensors

Sensors with no variation cannot provide useful predictive information
for the training data.

They are identified before model training.

In [7]:
feature_cols = [
    col for col in sensor_cols
    if col not in constant_sensors
]

X = train_df[feature_cols].copy()
y = train_df["RUL"].copy()

groups = train_df["unit"].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Number of groups:", groups.nunique())

Feature shape: (20631, 15)
Target shape: (20631,)
Number of groups: 100


#### Create ML Dataset

The machine-learning dataset is separated into:

- X: sensor features
- y: RUL target
- groups: engine identity

Engine identity is retained for leakage-safe validation.

In [8]:
print(
    "Missing feature values:",
    X.isna().sum().sum()
)

Missing feature values: 0


#### Check Feature Completeness

The selected raw sensor features contain no missing values.

Therefore, no missing-value treatment is required for the initial baseline
experiments.

In [9]:
from sklearn.model_selection import GroupKFold

group_kfold = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(
    group_kfold.split(X, y, groups=groups),
    start=1
):
    train_units = set(groups.iloc[train_idx])
    val_units = set(groups.iloc[val_idx])

    print(
        f"Fold {fold}: "
        f"train={len(train_idx)}, "
        f"validation={len(val_idx)}, "
        f"overlap={len(train_units & val_units)}"
    )

Fold 1: train=16511, validation=4120, overlap=0
Fold 2: train=16498, validation=4133, overlap=0
Fold 3: train=16505, validation=4126, overlap=0
Fold 4: train=16506, validation=4125, overlap=0
Fold 5: train=16504, validation=4127, overlap=0


#### Engine-Level Cross-Validation

GroupKFold is used so that observations from the same engine do not appear
in both training and validation data.

This prevents engine-level information leakage during model evaluation.

In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def evaluate_predictions(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return mae, rmse

#### Define Evaluation Metrics

Two primary metrics are used:

MAE measures the average absolute prediction error in RUL cycles.

RMSE penalizes larger prediction errors more strongly.

Both metrics will be used consistently across the models.

In [11]:
baseline_predictions = np.full(
    len(y),
    y.mean()
)

baseline_mae, baseline_rmse = evaluate_predictions(
    y,
    baseline_predictions
)

print("Baseline mean RUL:", y.mean())
print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)

Baseline mean RUL: 86.82928602588338
Baseline MAE: 36.96522596337408
Baseline RMSE: 41.67268886461814


#### Mean RUL Baseline

A simple baseline predicts the mean RUL for every observation.

This represents a model with no knowledge of the sensor measurements.

All machine-learning models should outperform this reference to demonstrate
useful predictive information.

In [12]:
def cross_validate_model(model, X, y, groups):
    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(
        GroupKFold(n_splits=5).split(X, y, groups),
        start=1
    ):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model.fit(X_train, y_train)

        predictions = model.predict(X_val)

        mae, rmse = evaluate_predictions(
            y_val,
            predictions
        )

        fold_results.append({
            "fold": fold,
            "MAE": mae,
            "RMSE": rmse
        })

    return fold_results

#### Create Reusable Model Evaluation

A common GroupKFold evaluation procedure is used for every classical
model.

This ensures that model comparisons are performed under the same
validation conditions.

In [13]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()

linear_results = cross_validate_model(
    linear_model,
    X,
    y,
    groups
)

linear_results

[{'fold': 1, 'MAE': 19.27880264774455, 'RMSE': np.float64(23.44996120323757)},
 {'fold': 2, 'MAE': 17.568784479012244, 'RMSE': np.float64(21.45894118335169)},
 {'fold': 3, 'MAE': 17.46328983711341, 'RMSE': np.float64(21.58724249269737)},
 {'fold': 4,
  'MAE': 15.819646485466928,
  'RMSE': np.float64(19.278415900259404)},
 {'fold': 5, 'MAE': 18.4303914701738, 'RMSE': np.float64(22.397099844226574)}]

#### Linear Regression

Linear Regression provides the first classical machine-learning model.

It assumes that the relationship between sensor measurements and RUL can
be approximated using a linear relationship.

In [14]:
linear_mae = np.mean([
    result["MAE"]
    for result in linear_results
])

linear_rmse = np.mean([
    result["RMSE"]
    for result in linear_results
])

print("Linear Regression MAE:", linear_mae)
print("Linear Regression RMSE:", linear_rmse)

Linear Regression MAE: 17.71218298390219
Linear Regression RMSE: 21.634332124754522


#### Linear Regression Results

The average MAE and RMSE across the five engine-level validation folds
are recorded for comparison with the baseline and later models.

In [15]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

knn_model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=5)
)

knn_results = cross_validate_model(
    knn_model,
    X,
    y,
    groups
)

knn_results

[{'fold': 1, 'MAE': 15.50873786407767, 'RMSE': np.float64(21.139282900642538)},
 {'fold': 2,
  'MAE': 13.964239051536413,
  'RMSE': np.float64(19.597007677090893)},
 {'fold': 3, 'MAE': 14.274697043141055, 'RMSE': np.float64(20.7714560941093)},
 {'fold': 4, 'MAE': 13.672339393939392, 'RMSE': np.float64(19.62339383120426)},
 {'fold': 5, 'MAE': 13.63683062757451, 'RMSE': np.float64(20.109755060833937)}]

#### KNN Regressor

KNN predicts RUL using observations with similar sensor measurements.

Feature scaling is applied because KNN depends on distances between
observations.

In [16]:
knn_mae = np.mean([
    result["MAE"]
    for result in knn_results
])

knn_rmse = np.mean([
    result["RMSE"]
    for result in knn_results
])

print("KNN MAE:", knn_mae)
print("KNN RMSE:", knn_rmse)

KNN MAE: 14.211368796053808
KNN RMSE: 20.248179112776185


#### KNN Results

The average validation MAE and RMSE are recorded for comparison with the
baseline and Linear Regression.

In [17]:
from sklearn.tree import DecisionTreeRegressor

tree_model = DecisionTreeRegressor(
    random_state=42,
    max_depth=10
)

tree_results = cross_validate_model(
    tree_model,
    X,
    y,
    groups
)

tree_results

[{'fold': 1,
  'MAE': 16.441123071280135,
  'RMSE': np.float64(22.432382205934356)},
 {'fold': 2, 'MAE': 15.332055773500274, 'RMSE': np.float64(21.16504351390437)},
 {'fold': 3,
  'MAE': 14.619978844299343,
  'RMSE': np.float64(21.159156964848464)},
 {'fold': 4, 'MAE': 14.197672933917294, 'RMSE': np.float64(20.19452952038568)},
 {'fold': 5,
  'MAE': 14.145624821496561,
  'RMSE': np.float64(20.756418376258612)}]

#### Decision Tree

Decision Tree Regression introduces nonlinear relationships between
sensor measurements and RUL.

A limited tree depth is used to avoid an unnecessarily complex model.

In [18]:
tree_mae = np.mean([
    result["MAE"]
    for result in tree_results
])

tree_rmse = np.mean([
    result["RMSE"]
    for result in tree_results
])

print("Decision Tree MAE:", tree_mae)
print("Decision Tree RMSE:", tree_rmse)

Decision Tree MAE: 14.947291088898723
Decision Tree RMSE: 21.141506116266292


#### Decision Tree Results

The average validation performance of the Decision Tree is recorded for
comparison with the previous models.

In [19]:
from sklearn.ensemble import RandomForestRegressor

forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

forest_results = cross_validate_model(
    forest_model,
    X,
    y,
    groups
)

forest_results

[{'fold': 1,
  'MAE': 15.005927552977113,
  'RMSE': np.float64(19.712448661780577)},
 {'fold': 2,
  'MAE': 13.562685732644882,
  'RMSE': np.float64(18.255381055730012)},
 {'fold': 3, 'MAE': 13.509269888943017, 'RMSE': np.float64(19.18775417322182)},
 {'fold': 4, 'MAE': 13.232768238422116, 'RMSE': np.float64(18.30642705200774)},
 {'fold': 5, 'MAE': 12.794493164974005, 'RMSE': np.float64(18.53769700825033)}]

#### Random Forest

Random Forest combines multiple decision trees to improve prediction
stability and capture nonlinear relationships.

In [20]:
forest_mae = np.mean([
    result["MAE"]
    for result in forest_results
])

forest_rmse = np.mean([
    result["RMSE"]
    for result in forest_results
])

print("Random Forest MAE:", forest_mae)
print("Random Forest RMSE:", forest_rmse)

Random Forest MAE: 13.621028915592225
Random Forest RMSE: 18.799941590198095


#### Random Forest Results

The average validation MAE and RMSE are recorded for comparison with the
previous classical models.

In [21]:
from sklearn.ensemble import GradientBoostingRegressor

gradient_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gradient_results = cross_validate_model(
    gradient_model,
    X,
    y,
    groups
)

gradient_results

[{'fold': 1, 'MAE': 15.40456127964868, 'RMSE': np.float64(19.81851904841066)},
 {'fold': 2, 'MAE': 13.842443730287838, 'RMSE': np.float64(18.14750548664429)},
 {'fold': 3, 'MAE': 13.821142268174976, 'RMSE': np.float64(19.41286734852505)},
 {'fold': 4, 'MAE': 13.512888132062503, 'RMSE': np.float64(18.26228374649034)},
 {'fold': 5, 'MAE': 13.303467674459593, 'RMSE': np.float64(18.74610599501851)}]

#### Gradient Boosting

Gradient Boosting builds sequential decision trees where each new tree
attempts to improve the previous model's errors.

This provides a stronger classical nonlinear model before considering
specialized boosting libraries.

In [22]:
gradient_mae = np.mean([
    result["MAE"]
    for result in gradient_results
])

gradient_rmse = np.mean([
    result["RMSE"]
    for result in gradient_results
])

print("Gradient Boosting MAE:", gradient_mae)
print("Gradient Boosting RMSE:", gradient_rmse)

Gradient Boosting MAE: 13.976900616926718
Gradient Boosting RMSE: 18.87745632501777


#### Gradient Boosting Results

The average validation performance of Gradient Boosting is recorded for
the final classical-model comparison.

In [23]:
results = pd.DataFrame({
    "Model": [
        "Mean Baseline",
        "Linear Regression",
        "KNN",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        baseline_mae,
        linear_mae,
        knn_mae,
        tree_mae,
        forest_mae,
        gradient_mae
    ],
    "RMSE": [
        baseline_rmse,
        linear_rmse,
        knn_rmse,
        tree_rmse,
        forest_rmse,
        gradient_rmse
    ]
})

results.sort_values("MAE")

,Model,MAE,RMSE
4,Random Forest,13.621029,18.799942
5,Gradient Boosting,13.976901,18.877456
2,KNN,14.211369,20.248179
3,Decision Tree,14.947291,21.141506
1,Linear Regression,17.712183,21.634332
0,Mean Baseline,36.965226,41.672689


#### Classical Model Comparison

The classical models are compared using the same engine-level validation
strategy and the same evaluation metrics.

Lower MAE and RMSE indicate better predictive performance.

In [24]:
best_model = results.loc[
    results["MAE"].idxmin()
]

print("Best model by MAE:")
print(best_model)

Best model by MAE:
Model    Random Forest
MAE          13.621029
RMSE         18.799942
Name: 4, dtype: object


#### Identify Best Classical Model

The model with the lowest validation MAE is identified as the strongest
classical baseline.

This decision is based on validation performance rather than model
complexity.

In [25]:
best_mae = best_model["MAE"]

improvement = (
    (baseline_mae - best_mae)
    / baseline_mae
) * 100

print("Baseline MAE:", baseline_mae)
print("Best model MAE:", best_mae)
print("MAE improvement (%):", improvement)

Baseline MAE: 36.96522596337408
Best model MAE: 13.621028915592225
MAE improvement (%): 63.15177694547782


#### Compare Against Mean Baseline

The best classical model is compared with the mean RUL baseline.

A lower MAE than the baseline indicates that the sensor features provide
useful information for RUL prediction.

In [26]:
results.sort_values(
    ["MAE", "RMSE"]
).reset_index(drop=True)

,Model,MAE,RMSE
0,Random Forest,13.621029,18.799942
1,Gradient Boosting,13.976901,18.877456
2,KNN,14.211369,20.248179
3,Decision Tree,14.947291,21.141506
4,Linear Regression,17.712183,21.634332
5,Mean Baseline,36.965226,41.672689


#### Model Comparison

| Model | MAE | RMSE |
|---|---:|---:|
| Random Forest | 13.62 | 18.80 |
| Gradient Boosting | 13.98 | 18.88 |
| KNN | 14.21 | 20.25 |
| Decision Tree | 14.95 | 21.14 |
| Linear Regression | 17.71 | 21.63 |
| Mean Baseline | 36.97 | 41.67 |

Random Forest achieved the lowest MAE and RMSE among the evaluated models.

#### Rank Classical Models

Models are ranked by MAE, with RMSE used as a secondary comparison.

This provides a clear performance hierarchy among the classical approaches.

Six baseline and classical machine-learning approaches were evaluated for
RUL prediction using engine-level cross-validation.

The models were compared using Mean Absolute Error (MAE) and Root Mean
Squared Error (RMSE).

The mean RUL predictor established a baseline MAE of approximately 36.97
cycles.

Among the classical models, Random Forest achieved the best performance
with an MAE of approximately 13.62 cycles and an RMSE of approximately
18.80 cycles.

Random Forest was followed by Gradient Boosting, KNN, Decision Tree, and
Linear Regression.

Compared with the mean baseline, Random Forest reduced MAE by approximately
63.15%.

In [27]:
from pathlib import Path

MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [28]:
results.to_csv(
    MODELS_DIR / "classical_model_results.csv",
    index=False
)

print("Results saved.")

Results saved.


In [29]:
import joblib

joblib.dump(
    forest_model,
    MODELS_DIR / "random_forest_rul.pkl"
)

print("Random Forest model saved.")

Random Forest model saved.


#### Save Model Results

The classical-model comparison is saved so that the results can be reused
without rerunning the complete experiment.

#### Decision

Classical machine-learning models were evaluated using engine-level
cross-validation.

The models were compared against a simple mean RUL baseline using MAE and
RMSE.

The best-performing classical model is selected based on validation
performance.

This model will serve as the reference point for later modeling phases.

#### Conclusion

Classical machine-learning models substantially outperformed the mean RUL
baseline.

Random Forest was the strongest model in this experiment, achieving an MAE
of 13.62 cycles and an RMSE of 18.80 cycles.

The approximately 63.15% reduction in MAE compared with the mean baseline
demonstrates that the sensor measurements contain substantial information
for predicting Remaining Useful Life.

Random Forest will therefore be carried forward as the current classical
ML reference model.



In [30]:
test_df.to_csv(
    "../data/test_FD001.csv",
    index=False
)

print("CSV saved successfully!")
print("Shape:", test_df.shape)

CSV saved successfully!
Shape: (13096, 26)
